In [41]:
import pandas as pd
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patches as patches
from adjustText import adjust_text
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import seaborn as sns
import glob
from matplotlib.ticker import ScalarFormatter, LogFormatter, FuncFormatter

from Functions import *
import MetricMapping

# Access the mappings:
type_mapping = MetricMapping.type_mapping
name_mapping = MetricMapping.name_mapping

In [42]:
datadir = '/scratch/hydro4/users/kv25483/MetricEvaluation/Data/IntermediateFiles/'
figdir = '/scratch/hydro4/users/kv25483/MetricEvaluation/Figures/'

In [43]:
resolutions = ['', '_dblnorm']

## Prepare the data
### Get the data

In [44]:
transformed_minmax_scaled = pd.read_csv(datadir+"/NotScaled_RawVsDMC_new.csv")

for res in resolutions:

    del transformed_minmax_scaled[f'min_intensity{res}']
    del transformed_minmax_scaled[f'frac_q1_wi{res}'] 
    del transformed_minmax_scaled[f'frac_q2_wi{res}']  
    del transformed_minmax_scaled[f'frac_q3_wi{res}']  
    del transformed_minmax_scaled[f'frac_q4_wi{res}']  
    del transformed_minmax_scaled[f'duration{res}']  
    
column_names = transformed_minmax_scaled.columns.str.replace('_log', '')
column_names = column_names.str.replace('_yj', '')
transformed_minmax_scaled.columns = column_names

/tmp/ipykernel_1884402/367090318.py:1: DtypeWarning: Columns (139) have mixed types. Specify dtype option on import or set low_memory=False.
  transformed_minmax_scaled = pd.read_csv(datadir+"/NotScaled_RawVsDMC_new.csv")


In [45]:
transformed_minmax_scaled['cv_dblnorm'] = transformed_minmax_scaled['std_dblnorm']/transformed_minmax_scaled['mean_intensity_dblnorm']

### Get a list of metrics

In [46]:
raw_cols = []
for col in transformed_minmax_scaled.columns:
    if not col.endswith('_DMC_10') and not col.endswith('dblnorm'):
        raw_cols.append(col)

#
categorical_metrics = ['3rd_ARR',  '3rd_rcg',  '3rd_w_peak', '4th_w_peak', '5th_w_peak', 'third_ppr', '3rd_w_most', 
                       '4th_w_most', '5th_w_most']
continuous_metrics = [metric for metric in list(type_mapping.keys()) if metric not in categorical_metrics]

### Calculate statistics on the data

In [47]:
summary_df = compute_metric_sensitivity_bynormalisation(df=transformed_minmax_scaled,
    continuous_metrics=continuous_metrics,
    categorical_metrics=categorical_metrics,
    resolutions =['dblnorm'])
summary_df["type2"] = summary_df["metric"].map(type_mapping)

missing column frac_q1_wi or frac_q1_wi_dblnorm
missing column centre_gravity_interpolated or centre_gravity_interpolated_dblnorm


/scratch/hydro4/users/kv25483/MetricEvaluation/Scripts/4. CompareNormalisation/Functions.py:214: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rank_corr, _ = spearmanr(x, y)


missing column I30 or I30_dblnorm
missing column lorenz_asymmetry or lorenz_asymmetry_dblnorm
missing column frac_q2_wi or frac_q2_wi_dblnorm
missing column frac_q3_wi or frac_q3_wi_dblnorm
missing column frac_q4_wi or frac_q4_wi_dblnorm


In [52]:
summary_df

,metric,resolution,type,rank_corr,val_diff,gini,type2
0,m3_wi,dblnorm,continuous,1.000000,7.782737e-04,0.338669,Mass timing
1,m4_wi,dblnorm,continuous,1.000000,8.509465e-04,0.355407,Mass timing
2,m5_wi,dblnorm,continuous,1.000000,5.115392e-04,0.259129,Mass timing
3,time_skewness,dblnorm,continuous,1.000000,1.402232e-12,3.930234,Mass timing
4,centre_gravity,dblnorm,continuous,1.000000,4.569399e-14,0.184344,Mass timing
5,T25,dblnorm,continuous,1.000000,3.100669e-04,0.407904,Mass timing
6,D50,dblnorm,continuous,1.000000,3.501176e-15,0.289085,Mass timing
7,T75,dblnorm,continuous,0.999995,4.189767e-04,0.188650,Mass timing
8,m1_wi,dblnorm,continuous,0.999999,9.819487e-04,0.598595,Peak timing
9,time_to_peak,dblnorm,continuous,NaN,1.947690e+02,0.000000,Peak timing


In [55]:
transformed_minmax_scaled['3rd_w_most_dblnorm']

0         1
1         1
2         2
3         1
4         2
         ..
240574    2
240575    0
240576    0
240577    1
240578    0
Name: 3rd_w_most_dblnorm, Length: 240579, dtype: int64

### Get versions of the data split by metric type`

In [49]:
df_continuous = summary_df[summary_df["type"] != "categorical"]
unique_metrics_continuous = df_continuous['metric'].unique().tolist()

df_categorical = summary_df[summary_df["type"] == "categorical"]
unique_metrics_categorical = df_categorical['metric'].unique().tolist()

## Plotting
### Scatter plot

In [51]:
import matplotlib.patheffects as pe

size = 800
labelsize = 25
marker = 'o'

# Points to label directly on main plot
mask_label   = ~((df_continuous["rank_corr"] > 0.05) & (df_continuous["val_diff"] <2))
# Everything else goes into the text list
mask_cluster = ~mask_label

df_label   = df_continuous[mask_label]
df_cluster = df_continuous[mask_cluster]

fig, ax = plt.subplots(figsize=(20, 20))

original_positions = []
face_colors = []
texts = []

# --- Plot all points ---
for _, row in df_continuous.iterrows():
    face_color = type_color_map_3[row['type2']][1]
    ax.scatter(
        row["rank_corr"], row["val_diff"],
        facecolor=face_color, edgecolor="black",
        marker=marker, s=size, alpha=1)

ax.tick_params(axis='both', which='major', labelsize=22)

# --- Label main plot points ---
for _, row in df_label.iterrows():
    face_color = type_color_map_3[row['type2']][1]
    # Add small random offset so adjust_text starts with spread-out positions
    jitter_x = np.random.uniform(-0.03, 0.03)
    jitter_y = np.random.uniform(-1.2, 1.2)
    txt = ax.text(
        row["rank_corr"] + jitter_x,
        row["val_diff"]  + jitter_y,
        name_mapping[row["metric"]],
        fontsize=labelsize, color='black',
        ha="center", va="center")
    txt.set_path_effects([pe.withStroke(linewidth=5, foreground="white")])
    texts.append(txt)
    original_positions.append((row["rank_corr"], row["val_diff"]))
    face_colors.append(face_color)

adjust_text(
    texts, ax=ax,
    only_move={'text': 'xy'},
    ensure_inside_axes=True,
    force_text=2.0,        # was not set (default ~0.2) — pushes text away from other text
    force_points=2.0,      # was not set (default ~0.2) — pushes text away from points
    expand_text=(2.0, 2.0),   # was not set — adds more breathing room around text
    expand_points=(2.0, 2.0), # was not set — adds more breathing room around points
    lim=500,               # more iterations to find better positions
    arrowprops=None)

for txt, (x0, y0), face_color in zip(texts, original_positions, face_colors):
    x1, y1 = txt.get_position()
    ax.annotate("", xy=(x0, y0), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="-", color=face_color, lw=1.5))

# --- Draw box around dense cluster and list names beside it ---
if len(df_cluster) > 0:
    pad = 0.01
    cx0 = df_cluster["rank_corr"].min() - pad
    cx1 = df_cluster["rank_corr"].max() + pad
    cy0 = df_cluster["val_diff"].min() # - pad
    cy1 = df_cluster["val_diff"].max()  + pad

    cluster_rect = patches.FancyBboxPatch(
        (cx0, cy0), cx1 - cx0, cy1 - cy0,
        boxstyle="round,pad=0.005",
        linewidth=2, edgecolor='black',
        facecolor='lightyellow', alpha=0.5, zorder=3)
    ax.add_patch(cluster_rect)

    # Build stacked name list with coloured dots
    cluster_names = "\n".join(
        [name_mapping[r["metric"]] for _, r in df_cluster.iterrows()])

#     cx0 =0.6
#     cx1 =0.8
#     cy0 =75
#     cy1 =100
    
    ax.annotate(
        cluster_names,
        xy=(cx1, (cy0 + cy1) / 2),
        xytext=(cx1 + 0.02, (cy0 + cy1) / 2),
        fontsize=labelsize - 4,
        va='center', ha='left',
        arrowprops=dict(arrowstyle="-", color='black', lw=1.5),
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow',
                  edgecolor='black', linewidth=1.2))

# --- Main axes ---
ax.set_xlabel("Spearman's ρ", fontsize=35)
ax.set_ylabel("sMAPE from 5m", fontsize=35)
ax.grid(True)

fig.suptitle("(a). Continuous metrics", fontsize=40)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

fig.savefig(
    figdir + "CompareNormalisation/Scatter_continuous_DMC10.png",
    dpi=300,
    facecolor='white',
    bbox_inches='tight')

StopIteration: 

Error in callback <function _draw_all_if_interactive at 0x7f60d3d88ee0> (for post_execute), with arguments args (),kwargs {}:


StopIteration: 

StopIteration: 

<Figure size 2000x2000 with 1 Axes>

In [ ]:
fig, ax = plt.subplots(figsize=(22, 20))  # Expanded canvas

marker = 'o'
size = 800
labelsize = 25
texts = []

for _, row in df_categorical.iterrows():
    # Scatter point
    ax.scatter(
        row["rank_corr"],
        row["val_diff"],
        facecolor=type_color_map_3[row['type2']][1],
        edgecolor='black',
        marker=marker,
        s=size,
        alpha=1)
    # Clean label name
    metric_name = name_mapping[row["metric"]]
    
    # Create text object (to be adjusted later)
    texts.append(
        ax.text(row["rank_corr"], row["val_diff"], metric_name, fontsize=labelsize))
ax.tick_params(axis='both', which='major', labelsize=22)
# Axes labels
ax.set_xlabel("Kendall’s τ", fontsize=30)
ax.set_ylabel("% Different from 5m", fontsize=30)
ax.grid(True)
# Adjust text to avoid overlaps
adjust_text(texts,
     ax=ax,
    arrowprops=dict(arrowstyle='-', color='gray', lw=0.5),
    expand_text=(1.05, 1.2),
    expand_points=(1.2, 1.4),
    only_move={'points': 'xy', 'text': 'xy'},
    force_points=0.5,
    force_text=1.2)

fig.suptitle("(b.) Categorical metrics", fontsize=40)
plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(figdir+ "CompareNormalisation/Scatter_categorical_dblnorm.png", dpi=300, facecolor='white', bbox_inches='tight')

## Legend

In [ ]:
legend_types = list(type_color_map_3.keys())

# Create figure
fig, ax = plt.subplots(figsize=(14, 1))

# Plot circles and labels
for i, label in enumerate(legend_types):
    
    color = type_color_map_3[label][1]
    
    # Handle long titles
    title = label.replace(" ", "\n", 1) if len(label) > 20 else label
    if "\n" not in title:
        title = title + "\n "  # blank second line for spacing
    
    ax.scatter(i, 0, s=900, color=color, edgecolor='black')
    ax.text(i + 0.15, 0, title, va='center', fontsize=16)

# Remove axes
ax.set_xlim(-0.5, len(legend_types)-0.2)
ax.set_ylim(-0.5, 0.5)
ax.axis("off")

plt.tight_layout()

# Save if needed
plt.savefig(figdir + "CompareNormalisation/Scatter_dblnorm_legend.png", dpi=300, bbox_inches="tight", facecolor="white")

plt.show()

### Combine figures into one

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import math

def combine_images_grid(image_paths, out_path,
                        ncols=6, pad=10, bg_color=(255,255,255),
                        dpi=300, labels=None):
    imgs = [Image.open(p).convert("RGB") for p in image_paths]
    widths, heights = zip(*(i.size for i in imgs))
    cell_w = max(widths)
    cell_h = max(heights)

    n = len(imgs)
    nrows = -(-n // ncols)   # ceiling division
    out_w = ncols*cell_w + pad*(ncols+1)
    out_h = nrows*cell_h + pad*(nrows+1)
    canvas = Image.new("RGB", (out_w, out_h), bg_color)

    for idx, img in enumerate(imgs):
        r = idx // ncols
        c = idx % ncols
        x = pad + c*(cell_w + pad)
        y = pad + r*(cell_h + pad)
        # center image in cell
        img_w, img_h = img.size
        paste_x = x + (cell_w - img_w)//2
        paste_y = y + (cell_h - img_h)//2
        canvas.paste(img, (paste_x, paste_y))

        # optional label under the image
        if labels:
            draw = ImageDraw.Draw(canvas)
            try:
                font = ImageFont.truetype("DejaVuSans.ttf", 12)
            except Exception:
                font = ImageFont.load_default()
            text = labels[idx]
            tw, th = draw.textsize(text, font=font)
            draw.rectangle([paste_x, paste_y+img_h - th - 6, paste_x + tw + 6, paste_y+img_h],
                           fill=(255,255,255))
            draw.text((paste_x+3, paste_y+img_h - th - 4), text, fill=(0,0,0), font=font)

    # save (PNG or PDF). dpi metadata set to (dpi,dpi)
    canvas.save(out_path, dpi=(dpi, dpi))
    
# Collect all PNGs (adjust pattern/order as needed)
files = sorted(glob.glob(figdir + "CompareNormalisation/Scatter_continuous_dblnorm.png")) + \
        sorted(glob.glob(figdir + "CompareNormalisation/Scatter_categorical_dblnorm.png")) 

combine_images_grid(files, figdir+"CompareNormalisation/Scatter_combined_dblnorm.png", ncols=2, pad=12, dpi=300)